In [2]:
import numpy as np
import pandas as pd
import pyblp
import scipy
import time
import statsmodels.api as sm
import matplotlib.pyplot as plt

Question 2.1

In [3]:
df = pd.read_csv('pset2_data.csv', index_col=0)

# outside share
s0 = 1 - df.groupby('market_ids')['shares'].transform('sum')
df['y'] = np.log(df['shares']) - np.log(s0)

# OLS
X_ols = df[['x', 'satellite', 'wired', 'prices']]
ols = sm.OLS(df['y'], X_ols).fit()
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.160
Model:                            OLS   Adj. R-squared:                  0.159
Method:                 Least Squares   F-statistic:                     152.5
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           1.91e-90
Time:                        13:37:47   Log-Likelihood:                -4702.0
No. Observations:                2400   AIC:                             9412.
Df Residuals:                    2396   BIC:                             9435.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
x              0.9419      0.122      7.752      0.0

Q 2.2

In [4]:
y = df['y'].values
X = df[['x', 'satellite', 'wired', 'prices']].values
Z = df[['x', 'satellite', 'wired', 'w']].values

In [5]:
# 2SLS
Pz = Z @ np.linalg.inv(Z.T @ Z) @ Z.T
X_hat = Pz @ X
beta_2sls = np.linalg.inv(X_hat.T @ X) @ (X_hat.T @ y)

# standard errors (use original X, not X_hat)
resid_2sls = y - X @ beta_2sls
n, k = X.shape
s2 = resid_2sls @ resid_2sls / (n - k)
se_2sls = np.sqrt(np.diag(s2 * np.linalg.inv(X_hat.T @ X)))

In [6]:
labels = ['x (beta1)', 'satellite (beta2)', 'wired (beta3)', 'prices (alpha)']
print("=== 2SLS Logit Results ===")
print(f"{'Variable':<22} {'Coeff':>10} {'Std.Err':>10}")
for i, lb in enumerate(labels):
    print(f"{lb:<22} {beta_2sls[i]:>10.4f} {se_2sls[i]:>10.4f}")

# first-stage partial F-stat
exog = df[['x', 'satellite', 'wired']].values
p = df['prices'].values
ssr_full = np.sum((p - Z @ np.linalg.lstsq(Z, p, rcond=None)[0])**2)
ssr_rest = np.sum((p - exog @ np.linalg.lstsq(exog, p, rcond=None)[0])**2)
F_stat = ((ssr_rest - ssr_full) / 1) / (ssr_full / (n - Z.shape[1]))
print(f"\nFirst-stage partial F-stat: {F_stat:.2f}")

=== 2SLS Logit Results ===
Variable                    Coeff    Std.Err
x (beta1)                  1.0677     0.1429
satellite (beta2)          4.7245     1.0479
wired (beta3)              4.7038     1.0383
prices (alpha)            -2.3569     0.3954

First-stage partial F-stat: 51.66


Q 2.3

In [7]:
# within-group share
df['group'] = df['satellite'].astype(int)
s_g = df.groupby(['market_ids', 'group'])['shares'].transform('sum')
df['within_share'] = df['shares'] / s_g
df['ln_within'] = np.log(df['within_share'])

# allow different sigma per nest
df['sat_ln_within'] = df['satellite'] * df['ln_within']
df['wired_ln_within'] = df['wired'] * df['ln_within']

# BLP-style IV: rival's x and w within same nest
df['rival_x_nest'] = df.groupby(['market_ids', 'group'])['x'].transform('sum') - df['x']
df['rival_w_nest'] = df.groupby(['market_ids', 'group'])['w'].transform('sum') - df['w']

# regressors and instruments
X = df[['x', 'satellite', 'wired', 'prices', 'sat_ln_within', 'wired_ln_within']].values
y = df['y'].values
Z = df[['x', 'satellite', 'wired', 'w', 'rival_x_nest', 'rival_w_nest']].values

In [8]:
# 2SLS
Pz = Z @ np.linalg.inv(Z.T @ Z) @ Z.T
X_hat = Pz @ X
beta_nl = np.linalg.inv(X_hat.T @ X) @ (X_hat.T @ y)

resid_nl = y - X @ beta_nl
n, k = X.shape
s2 = resid_nl @ resid_nl / (n - k)
se_nl = np.sqrt(np.diag(s2 * np.linalg.inv(X_hat.T @ X)))

labels = ['x (beta1)', 'satellite (beta2)', 'wired (beta3)',
          'prices (alpha)', 'sigma_sat', 'sigma_wired']
print("=== Nested Logit 2SLS ===")
print(f"{'Variable':<22} {'Coeff':>10} {'Std.Err':>10}")
for i, lb in enumerate(labels):
    print(f"{lb:<22} {beta_nl[i]:>10.4f} {se_nl[i]:>10.4f}")

=== Nested Logit 2SLS ===
Variable                    Coeff    Std.Err
x (beta1)                  0.7867     0.2936
satellite (beta2)          3.1652     1.9209
wired (beta3)              6.1489     3.3219
prices (alpha)            -2.2740     0.7872
sigma_sat                 -1.1728     2.0448
sigma_wired                1.2370     1.3706


Q 2.4

In [9]:
# use estimates from Q2.3
alpha = beta_nl[3]
sigma_sat = beta_nl[4]
sigma_wired = beta_nl[5]
df['sigma_g'] = np.where(df['satellite'] == 1, sigma_sat, sigma_wired)

# own-price elasticity
df['eta_own'] = alpha * df['prices'] * (
    1 / (1 - df['sigma_g'])
    - df['sigma_g'] / (1 - df['sigma_g']) * df['within_share']
    - df['shares']
)

In [10]:
T, J = 600, 4
eta_avg = df['eta_own'].values.reshape((T, J)).mean(axis=0)
true_eta = pd.read_csv('true_ownprice_elasticities.csv', index_col=0).values.flatten()

print("=== Own-Price Elasticities ===")
print(f"{'':>12} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
print(f"{'Estimated':>12} {eta_avg[0]:>10.4f} {eta_avg[1]:>10.4f} {eta_avg[2]:>10.4f} {eta_avg[3]:>10.4f}")
print(f"{'True':>12} {true_eta[0]:>10.4f} {true_eta[1]:>10.4f} {true_eta[2]:>10.4f} {true_eta[3]:>10.4f}")

# diversion ratios
shares = df['shares'].values.reshape((T, J))
groups = df['group'].values.reshape((T, J))
within = df['within_share'].values.reshape((T, J))
sigma_arr = df['sigma_g'].values.reshape((T, J))

div_ratio = np.zeros((T, J, J))
for j in range(J):
    sig_j = sigma_arr[:, j]
    ds_j = alpha * shares[:, j] * (
        1 / (1 - sig_j) - sig_j / (1 - sig_j) * within[:, j] - shares[:, j]
    )
    for k in range(J):
        if k == j:
            continue
        same = (groups[:, j] == groups[:, k])
        ds_k = -alpha * shares[:, k] * np.where(
            same,
            sig_j / (1 - sig_j) * within[:, j] + shares[:, j],
            shares[:, j]
        )
        div_ratio[:, j, k] = -ds_k / ds_j
    div_ratio[:, j, j] = 1 - div_ratio[:, j, :].sum(axis=1)

div_avg = div_ratio.mean(axis=0)
true_div = pd.read_csv('true_diversionratio.csv', index_col=0).values

print("\n=== Estimated Diversion Ratios ===")
print(f"{'':>6} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
for j in range(J):
    print(f"{'J'+str(j+1):>6} {div_avg[j,0]:>10.4f} {div_avg[j,1]:>10.4f} {div_avg[j,2]:>10.4f} {div_avg[j,3]:>10.4f}")

print("\n=== True Diversion Ratios ===")
print(f"{'':>6} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
for j in range(J):
    print(f"{'J'+str(j+1):>6} {true_div[j,0]:>10.4f} {true_div[j,1]:>10.4f} {true_div[j,2]:>10.4f} {true_div[j,3]:>10.4f}")

=== Own-Price Elasticities ===
                     J1         J2         J3         J4
   Estimated    -3.3055    -3.3520    11.2183    11.3695
        True    -4.2500    -4.3714    -4.2711    -4.2576

=== Estimated Diversion Ratios ===
               J1         J2         J3         J4
    J1     0.5004    -0.1472     0.3216     0.3252
    J2    -0.1585     0.5036     0.3247     0.3301
    J3     0.2183    -0.8854    -0.1990     1.8661
    J4    -0.1710     0.0622     1.3125    -0.2037

=== True Diversion Ratios ===
               J1         J2         J3         J4
    J1     0.3338     0.2170     0.2204     0.2288
    J2     0.2194     0.3370     0.2216     0.2220
    J3     0.2193     0.2185     0.3350     0.2272
    J4     0.2246     0.2150     0.2232     0.3371


Q 3.5

In [11]:
pyblp.options.digits = 3
pyblp.options.verbose = False

# Formulations
X1 = pyblp.Formulation('0 + x + satellite + wired + prices')
X2 = pyblp.Formulation('0 + satellite + wired')
X3 = pyblp.Formulation('0 + satellite + wired + w')  # avoid collinearity with constant

# Instruments
df['group'] = df['satellite'].astype(int)
df['rival_x'] = df.groupby('market_ids')['x'].transform('sum') - df['x']
df['rival_w'] = df.groupby('market_ids')['w'].transform('sum') - df['w']
df['rival_x_nest'] = df.groupby(['market_ids','group'])['x'].transform('sum') - df['x']
df['rival_w_nest'] = df.groupby(['market_ids','group'])['w'].transform('sum') - df['w']

df['demand_instruments0'] = df['w']
df['demand_instruments1'] = df['rival_x']
df['demand_instruments2'] = df['rival_w']
df['demand_instruments3'] = df['rival_x_nest']
df['demand_instruments4'] = df['rival_w_nest']

df['supply_instruments0'] = df['x']
df['supply_instruments1'] = df['rival_x']
df['supply_instruments2'] = df['rival_w']
df['supply_instruments3'] = df['rival_x_nest']
df['supply_instruments4'] = df['rival_w_nest']

integration = pyblp.Integration('product', size=7)
sigma0 = np.array([[1.0, 0], [0, 1.0]])
beta0 = np.full((4, 1), np.nan)
beta0[3] = -2.0  # initial guess for alpha

In [12]:
# (a) Demand only
problem_a = pyblp.Problem(product_formulations=(X1, X2), product_data=df, integration=integration)
results_a = problem_a.solve(sigma=sigma0, method='1s',
    optimization=pyblp.Optimization('l-bfgs-b', {'maxfun': 5000}))
print(results_a)

Problem Results Summary:
GMM   Objective    Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ---------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 1    +6.37E+00    +3.82E-06       +8.10E-01        +5.84E+00        0        +3.39E+02          +5.80E+04    

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:32       Yes          10           12          61308       188169   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
 Sigma:     satellite      wired   
---------  -----------  -----------
satellite   +2.05E+00              
           (+2.47E+00)           

In [15]:
# (b) Demand + Supply
problem_b = pyblp.Problem(product_formulations=(X1, X2, X3), product_data=df, integration=integration)
results_b = problem_b.solve(sigma=sigma0, beta=beta0, method='1s',
    optimization=pyblp.Optimization('l-bfgs-b', {'maxfun': 5000}))
print(results_b)

Problem Results Summary:
GMM   Objective    Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ---------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 1    +9.46E+00    +5.68E-06       +7.28E-01        +5.86E+01        0        +3.39E+02          +6.31E+04    

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:44       Yes          12           17          85663       262814   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
 Sigma:     satellite      wired   
---------  -----------  -----------
satellite   +1.94E+00              
           (+2.50E+00)           

In [19]:
# (c) Optimal IV
opt_iv = results_b.compute_optimal_instruments()
opt_problem = opt_iv.to_problem()
results_c = opt_problem.solve(sigma=results_b.sigma, beta=beta0, method='1s',
    optimization=pyblp.Optimization('l-bfgs-b', {'maxfun': 5000}))
print(results_c)

Problem Results Summary:
GMM   Objective    Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ---------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 1    +9.93E+00    +6.34E-07       +6.70E+00        +5.59E+01        0        +4.48E+05          +1.76E+04    

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:44       Yes          11           13          67163       205702   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
 Sigma:     satellite      wired   
---------  -----------  -----------
satellite   +1.80E+00              
           (+1.08E+00)           

Q 3.6

In [21]:
T, J = 600, 4

# own-price elasticities
elasticities = results_c.compute_elasticities()
elast_avg = elasticities.reshape((T, J, J)).mean(axis=0)

true_eta = pd.read_csv('true_ownprice_elasticities.csv', index_col=0).values.flatten()

print("=== Own-Price Elasticities ===")
print(f"{'':>12} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
print(f"{'Estimated':>12} {elast_avg[0,0]:>10.4f} {elast_avg[1,1]:>10.4f} {elast_avg[2,2]:>10.4f} {elast_avg[3,3]:>10.4f}")
print(f"{'True':>12} {true_eta[0]:>10.4f} {true_eta[1]:>10.4f} {true_eta[2]:>10.4f} {true_eta[3]:>10.4f}")

# diversion ratios
diversion = results_c.compute_diversion_ratios()
div_avg = diversion.reshape((T, J, J)).mean(axis=0)

true_div = pd.read_csv('true_diversionratio.csv', index_col=0).values

print("\n=== Estimated Diversion Ratios ===")
print(f"{'':>6} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
for j in range(J):
    print(f"{'J'+str(j+1):>6} {div_avg[j,0]:>10.4f} {div_avg[j,1]:>10.4f} {div_avg[j,2]:>10.4f} {div_avg[j,3]:>10.4f}")

print("\n=== True Diversion Ratios ===")
print(f"{'':>6} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
for j in range(J):
    print(f"{'J'+str(j+1):>6} {true_div[j,0]:>10.4f} {true_div[j,1]:>10.4f} {true_div[j,2]:>10.4f} {true_div[j,3]:>10.4f}")

=== Own-Price Elasticities ===
                     J1         J2         J3         J4
   Estimated    -5.2736    -5.4717    -5.3342    -5.3405
        True    -4.2500    -4.3714    -4.2711    -4.2576

=== Estimated Diversion Ratios ===
               J1         J2         J3         J4
    J1     0.3328     0.3667     0.1465     0.1540
    J2     0.3733     0.3358     0.1454     0.1455
    J3     0.1465     0.1456     0.3353     0.3726
    J4     0.1487     0.1395     0.3729     0.3388

=== True Diversion Ratios ===
               J1         J2         J3         J4
    J1     0.3338     0.2170     0.2204     0.2288
    J2     0.2194     0.3370     0.2216     0.2220
    J3     0.2193     0.2185     0.3350     0.2272
    J4     0.2246     0.2150     0.2232     0.3371


Q 3.7

In [22]:
n_boot = 50
T, J = 600, 4
markets = df['market_ids'].unique()
np.random.seed(42)

# use results_c from Q3.5
sigma_start = results_c.sigma
beta_start = results_c.beta.copy()

boot_div = []
t0 = time.time()

for b in range(n_boot):
    boot_mkt = np.random.choice(markets, size=T, replace=True)
    frames = []
    for new_id, old_id in enumerate(boot_mkt):
        chunk = df[df['market_ids'] == old_id].copy()
        chunk['market_ids'] = new_id
        frames.append(chunk)
    bdf = pd.concat(frames, ignore_index=True)

    # recompute instruments
    bdf['group'] = bdf['satellite'].astype(int)
    bdf['rival_x'] = bdf.groupby('market_ids')['x'].transform('sum') - bdf['x']
    bdf['rival_w'] = bdf.groupby('market_ids')['w'].transform('sum') - bdf['w']
    bdf['rival_x_nest'] = bdf.groupby(['market_ids','group'])['x'].transform('sum') - bdf['x']
    bdf['rival_w_nest'] = bdf.groupby(['market_ids','group'])['w'].transform('sum') - bdf['w']
    bdf['demand_instruments0'] = bdf['w']
    bdf['demand_instruments1'] = bdf['rival_x']
    bdf['demand_instruments2'] = bdf['rival_w']
    bdf['demand_instruments3'] = bdf['rival_x_nest']
    bdf['demand_instruments4'] = bdf['rival_w_nest']
    bdf['supply_instruments0'] = bdf['x']
    bdf['supply_instruments1'] = bdf['rival_x']
    bdf['supply_instruments2'] = bdf['rival_w']
    bdf['supply_instruments3'] = bdf['rival_x_nest']
    bdf['supply_instruments4'] = bdf['rival_w_nest']

    try:
        prob = pyblp.Problem(product_formulations=(X1, X2, X3), product_data=bdf, integration=integration)
        res = prob.solve(sigma=sigma_start, beta=beta_start, method='1s',
            optimization=pyblp.Optimization('l-bfgs-b', {'maxfun': 1000}))
        div_avg = res.compute_diversion_ratios().reshape((T, J, J)).mean(axis=0)
        boot_div.append(div_avg)
        print(f"Boot {b+1}/{n_boot} ({time.time()-t0:.0f}s)")
    except:
        print(f"Boot {b+1}/{n_boot} FAILED")

boot_div = np.array(boot_div)
lo = np.percentile(boot_div, 2.5, axis=0)
hi = np.percentile(boot_div, 97.5, axis=0)

true_div = pd.read_csv('true_diversionratio.csv', index_col=0).values

print("\n=== Bootstrap 95% CI for Diversion Ratios ===")
print(f"{'':>6} {'J1':>20} {'J2':>20} {'J3':>20} {'J4':>20}")
for j in range(J):
    row = f"{'J'+str(j+1):>6} "
    for k in range(J):
        row += f" [{lo[j,k]:.3f}, {hi[j,k]:.3f}]  "
    print(row)

# coverage check
print("\n=== Coverage ===")
for j in range(J):
    for k in range(J):
        inside = lo[j,k] <= true_div[j,k] <= hi[j,k]
        print(f"  D({j+1},{k+1}): [{lo[j,k]:.3f},{hi[j,k]:.3f}], true={true_div[j,k]:.3f} -> {'IN' if inside else 'OUT'}")

Boot 1/50 (140s)
Boot 2/50 (212s)
Boot 3/50 (401s)
Boot 4/50 (515s)
Boot 5/50 (693s)
Boot 6/50 (893s)
Boot 7/50 (1099s)
Boot 8/50 (1186s)
Boot 9/50 (1404s)
Boot 10/50 (1603s)
Boot 11/50 (1773s)
Boot 12/50 (1913s)
Boot 13/50 (2016s)
Boot 14/50 (2164s)
Boot 15/50 (2310s)
Boot 16/50 (2442s)
Boot 17/50 (2555s)
Boot 18/50 (2763s)
Boot 19/50 (2943s)
Boot 20/50 (3110s)
Boot 21/50 (3269s)
Boot 22/50 (3355s)
Boot 23/50 (3482s)
Boot 24/50 (3584s)
Boot 25/50 (3788s)
Boot 26/50 (3955s)
Boot 27/50 (4072s)
Boot 28/50 (4309s)
Boot 29/50 (4456s)
Boot 30/50 (4529s)
Boot 31/50 (4659s)
Boot 32/50 (4803s)
Boot 33/50 (4959s)
Boot 34/50 (5131s)
Boot 35/50 (5257s)
Boot 36/50 (5348s)
Boot 37/50 (5537s)
Boot 38/50 (5676s)
Boot 39/50 (5832s)
Boot 40/50 (5950s)
Boot 41/50 (6088s)
Boot 42/50 (6235s)
Boot 43/50 (6382s)
Boot 44/50 (6470s)
Boot 45/50 (6544s)
Boot 46/50 (6790s)
Boot 47/50 (6952s)
Boot 48/50 (7080s)
Boot 49/50 (7247s)
Boot 50/50 (7315s)

=== Bootstrap 95% CI for Diversion Ratios ===
                  

Q 4.9

In [24]:
# Merger simulation
T, J = 600, 4

# recover marginal costs
costs = results_c.compute_costs()

# merger
df['merger_ids'] = df['firm_ids'].replace({1: 0})

# post-merger equilibrium prices
post_prices = results_c.compute_prices(
    firm_ids=df['merger_ids'].values,
    costs=costs
).flatten()

pre_prices = df['prices'].values
pct_change = (post_prices / pre_prices - 1) * 100

print("=== Merger Firms ===")
print(f"{'':>20} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
print(f"{'Pre-merger':>20} {pre_prices.reshape((T,J)).mean(0)[0]:>10.4f} {pre_prices.reshape((T,J)).mean(0)[1]:>10.4f} {pre_prices.reshape((T,J)).mean(0)[2]:>10.4f} {pre_prices.reshape((T,J)).mean(0)[3]:>10.4f}")
print(f"{'Post-merger':>20} {post_prices.reshape((T,J)).mean(0)[0]:>10.4f} {post_prices.reshape((T,J)).mean(0)[1]:>10.4f} {post_prices.reshape((T,J)).mean(0)[2]:>10.4f} {post_prices.reshape((T,J)).mean(0)[3]:>10.4f}")
print(f"{'% change':>20} {pct_change.reshape((T,J)).mean(0)[0]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[1]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[2]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[3]:>10.2f}%")

=== Merger Firms ===
                             J1         J2         J3         J4
          Pre-merger     2.6458     2.7293     2.6594     2.6675
         Post-merger     2.9442     3.0326     2.6667     2.6733
            % change      12.14%      11.85%       0.31%       0.24%


Q 4.10

In [26]:
# Merger of Firms 1 & 3
merger_ids_13 = df['firm_ids'].replace({2: 0}).values  # firm 0 and firm 2 -> same owner
post_13 = results_c.compute_prices(firm_ids=merger_ids_13, costs=costs).flatten()
pct_13 = (post_13 / pre_prices - 1) * 100

T, J = 600, 4
avg_pct_12 = pct_change.reshape((T, J)).mean(axis=0)  # from Q4.9
avg_pct_13 = pct_13.reshape((T, J)).mean(axis=0)

print("=== Comparison of Merger-Induced Price Changes ===")
print(f"{'':>25} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
print(f"{'Firms 1&2 (sat-sat)':>25} {avg_pct_12[0]:>10.2f}% {avg_pct_12[1]:>10.2f}% {avg_pct_12[2]:>10.2f}% {avg_pct_12[3]:>10.2f}%")
print(f"{'Firms 1&3 (sat-wired)':>25} {avg_pct_13[0]:>10.2f}% {avg_pct_13[1]:>10.2f}% {avg_pct_13[2]:>10.2f}% {avg_pct_13[3]:>10.2f}%")

=== Comparison of Merger-Induced Price Changes ===
                                  J1         J2         J3         J4
      Firms 1&2 (sat-sat)      12.14%      11.85%       0.31%       0.24%
    Firms 1&3 (sat-wired)       3.81%       0.35%       3.87%       0.32%


Q 4.12

In [27]:
# Merger with 15% cost reduction
merger_ids = df['firm_ids'].replace({1: 0}).values

merger_costs = costs.copy()
merger_costs[merger_ids == 0] = 0.85 * merger_costs[merger_ids == 0]

post_eff = results_c.compute_prices(firm_ids=merger_ids, costs=merger_costs).flatten()
pct_eff = (post_eff / pre_prices - 1) * 100

T, J = 600, 4
print("=== Price Changes ===")
print(f"{'':>25} {'J1':>10} {'J2':>10} {'J3':>10} {'J4':>10}")
print(f"{'No efficiencies':>25} {pct_change.reshape((T,J)).mean(0)[0]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[1]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[2]:>10.2f}% {pct_change.reshape((T,J)).mean(0)[3]:>10.2f}%")
print(f"{'With 15% cost cut':>25} {pct_eff.reshape((T,J)).mean(0)[0]:>10.2f}% {pct_eff.reshape((T,J)).mean(0)[1]:>10.2f}% {pct_eff.reshape((T,J)).mean(0)[2]:>10.2f}% {pct_eff.reshape((T,J)).mean(0)[3]:>10.2f}%")

# consumer surplus
cs_pre = results_c.compute_consumer_surpluses()
cs_post_no = results_c.compute_consumer_surpluses(prices=post_prices.reshape(-1,1))
cs_post_eff = results_c.compute_consumer_surpluses(prices=post_eff.reshape(-1,1))

print(f"\nDelta CS per market (no eff):      {(cs_post_no - cs_pre).mean():.4f}")
print(f"Delta CS per market (15% cost cut): {(cs_post_eff - cs_pre).mean():.4f}")

=== Price Changes ===
                                  J1         J2         J3         J4
          No efficiencies      12.14%      11.85%       0.31%       0.24%
        With 15% cost cut       4.67%       4.11%       0.02%      -0.04%

Delta CS per market (no eff):      -0.0717
Delta CS per market (15% cost cut): -0.0162


Q 4.13

In [29]:
# Total welfare
cs_pre = results_c.compute_consumer_surpluses().flatten()
cs_post = results_c.compute_consumer_surpluses(prices=post_eff.reshape(-1,1)).flatten()
delta_cs = cs_post - cs_pre

post_shares = results_c.compute_shares(prices=post_eff.reshape(-1,1)).flatten()
pre_shares = df['shares'].values

pre_ps = ((pre_prices - costs.flatten()) * pre_shares).reshape((T,J)).sum(axis=1)
post_ps = ((post_eff - merger_costs.flatten()) * post_shares).reshape((T,J)).sum(axis=1)
delta_ps = post_ps - pre_ps

delta_tw = delta_cs + delta_ps

print(f"Avg delta CS per market: {delta_cs.mean():.4f}")
print(f"Avg delta PS per market: {delta_ps.mean():.4f}")
print(f"Avg delta TW per market: {delta_tw.mean():.4f}")

Avg delta CS per market: -0.0162
Avg delta PS per market: 0.1127
Avg delta TW per market: 0.0966
